# Stage 4 — Model 1: Extremely Randomized Trees (`ExtraTreesClassifier`)
**Project:** 10-Year Coronary Heart Disease (CHD) Risk Prediction (Framingham Heart Study)  
**Author:** Saif Asaad Naeem | Al-Maaqal University — Department of AI Engineering

---
### Notebook Roadmap:
1. **Environment Setup & Data Loading:** Load `framingham_balanced.csv` (`6,147` samples, `20` clinical features) and perform a stratified 80%/20% train/test split (`random_state=42`).
2. **Stage 1 — Baseline Model Evaluation:** Train and evaluate the default `ExtraTreesClassifier` under 5-Fold Stratified Cross-Validation.
3. **Stage 2 — Continuous Exploration via `RandomizedSearchCV ()`:** Sample hyperparameters from continuous and discrete probability distributions (`randint(...)`, `uniform(...)`).
4. **Stage 3 — Exhaustive Exploitation via `GridSearchCV []`:** Construct a focused parameter lattice `[...]` around the optimal `RandomizedSearchCV` neighborhood.
5. **Final Model Comparison, Serialization & Diagnostic Plots:** Export `best_model1_extratrees.pkl` and plot the test Confusion Matrix and ROC curve.

### 1. Environment Setup and Data Loading
Load required libraries, verify `framingham_balanced.csv`, and configure the unified 5-Fold Stratified Cross-Validation evaluator.

In [ ]:
import os
import joblib
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import randint, uniform

from sklearn.ensemble import ExtraTreesClassifier, RandomForestRegressor
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV, GridSearchCV, cross_validate
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, RocCurveDisplay

# Automatic fallback generator if framingham_balanced.csv is not yet present in the working directory
if not os.path.exists('framingham_balanced.csv'):
    df_raw = pd.read_csv('framingham.csv')
    X_r, y_r = df_raw.drop('TenYearCHD', axis=1), df_raw['TenYearCHD'].values
    mf = IterativeImputer(estimator=RandomForestRegressor(n_estimators=20, max_depth=8, random_state=42), max_iter=5, random_state=42)
    d = pd.DataFrame(mf.fit_transform(X_r), columns=X_r.columns)
    d['education'] = X_r['education'].fillna(X_r['education'].mode()[0])
    d['BPMeds'] = X_r['BPMeds'].copy()
    d.loc[(d['BPMeds'].isnull()) & (d['prevalentHyp'] == 0), 'BPMeds'] = 0.0
    d['BPMeds'] = d['BPMeds'].fillna(0.0)
    d['pulse_pressure'] = d['sysBP'] - d['diaBP']
    d['MAP'] = d['diaBP'] + (d['sysBP'] - d['diaBP']) / 3.0
    d['sys_dia_ratio'] = d['sysBP'] / (d['diaBP'] + 1e-5)
    d['rate_pressure_product'] = (d['sysBP'] * d['heartRate']) / 1000.0
    d['age_sysBP_index'] = (d['age'] * d['sysBP']) / 100.0
    d['age_glucose_index'] = (d['age'] * d['glucose']) / 100.0
    d['bmi_chol_index'] = (d['BMI'] * d['totChol']) / 1000.0
    d['log_gluc_sysBP'] = np.log1p(d['glucose']) * np.log1p(d['sysBP'])
    sel_20 = ['male', 'age', 'cigsPerDay', 'BPMeds', 'prevalentStroke', 'prevalentHyp', 'totChol', 'sysBP', 'diaBP', 'BMI', 'heartRate', 'glucose', 'pulse_pressure', 'MAP', 'sys_dia_ratio', 'rate_pressure_product', 'age_sysBP_index', 'age_glucose_index', 'bmi_chol_index', 'log_gluc_sysBP']
    X_sc = StandardScaler().fit_transform(d[sel_20].values)
    rng = np.random.RandomState(42)
    X_maj, X_min = X_sc[y_r == 0], X_sc[y_r == 1]
    n_syn = len(X_maj) - len(X_min)
    _, ind = NearestNeighbors(n_neighbors=6).fit(X_min).kneighbors(X_min)
    syn = [X_min[i := rng.randint(0, len(X_min))] + rng.uniform(0, 1) * (X_min[ind[i, rng.randint(1, 6)]] - X_min[i]) for _ in range(n_syn)]
    X_sm, y_sm = np.vstack([X_sc, np.array(syn)]), np.hstack([y_r, np.ones(n_syn, dtype=y_r.dtype)])
    _, ind_e = NearestNeighbors(n_neighbors=4).fit(X_sm).kneighbors(X_sm)
    keep = np.array([y_sm[i] == (1 if np.sum(y_sm[ind_e[i, 1:]] == 1) >= 2 else 0) for i in range(len(X_sm))])
    df_bal = pd.DataFrame(X_sm[keep], columns=sel_20)
    df_bal['TenYearCHD'] = y_sm[keep]
    df_bal.to_csv('framingham_balanced.csv', index=False)

df = pd.read_csv('framingham_balanced.csv')
X = df.drop('TenYearCHD', axis=1).values
y = df['TenYearCHD'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring_metrics = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']

def evaluate_model(model, stage_name):
    cv_res = cross_validate(model, X_train, y_train, cv=skf, scoring=scoring_metrics, n_jobs=-1)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    return {
        'Stage': stage_name,
        'CV_Recall (%)': round(cv_res['test_recall'].mean() * 100, 2),
        'CV_Precision (%)': round(cv_res['test_precision'].mean() * 100, 2),
        'CV_F1 (%)': round(cv_res['test_f1'].mean() * 100, 2),
        'CV_ROC_AUC (%)': round(cv_res['test_roc_auc'].mean() * 100, 2),
        'CV_Accuracy (%)': round(cv_res['test_accuracy'].mean() * 100, 2),
        'Test_Recall (%)': round(recall_score(y_test, y_pred) * 100, 2),
        'Test_Precision (%)': round(precision_score(y_test, y_pred) * 100, 2),
        'Test_F1 (%)': round(f1_score(y_test, y_pred) * 100, 2),
        'Test_ROC_AUC (%)': round(roc_auc_score(y_test, y_prob) * 100, 2),
        'Test_Accuracy (%)': round(accuracy_score(y_test, y_pred) * 100, 2)
    }

print(f"Dataset Loaded: Train = {X_train.shape}, Test = {X_test.shape}")

### 2. Stage 1: Baseline `ExtraTreesClassifier` Evaluation
Evaluate the untuned default `ExtraTreesClassifier` model.

In [ ]:
base_et = ExtraTreesClassifier(random_state=42, n_jobs=-1)
res_base = evaluate_model(base_et, '1. Baseline ExtraTrees')
print("Baseline Evaluation:", res_base)

### 3. Stage 2: Broad Exploration via `RandomizedSearchCV ()`
Use continuous and discrete probability distributions enclosed in parentheses `()` to explore the hyperparameter space.

In [ ]:
param_dist_et = {
    'n_estimators': randint(150, 500),
    'max_depth': randint(12, 35),
    'min_samples_split': randint(2, 10),
    'min_samples_leaf': randint(1, 5),
    'max_features': uniform(0.4, 0.6)
}

rand_et = RandomizedSearchCV(
    estimator=ExtraTreesClassifier(random_state=42, n_jobs=-1),
    param_distributions=param_dist_et,
    n_iter=25,
    scoring='f1',
    cv=skf,
    random_state=42,
    n_jobs=-1
)
rand_et.fit(X_train, y_train)
best_r = rand_et.best_params_
print("Best Parameters from RandomizedSearchCV ():", best_r)
res_rand = evaluate_model(rand_et.best_estimator_, '2. RandomizedSearchCV ()')

### 4. Stage 3: Fine-Grained Exploitation via `GridSearchCV []`
Bracket the optimal parameters found by `RandomizedSearchCV` inside explicit lists `[]` for exhaustive local grid search.

In [ ]:
param_grid_et = {
    'n_estimators': [max(50, best_r['n_estimators'] - 40), best_r['n_estimators'], best_r['n_estimators'] + 40],
    'max_depth': [max(5, best_r['max_depth'] - 3), best_r['max_depth'], best_r['max_depth'] + 3],
    'min_samples_split': [max(2, best_r['min_samples_split'] - 1), best_r['min_samples_split'], best_r['min_samples_split'] + 1],
    'min_samples_leaf': [max(1, best_r['min_samples_leaf']), best_r['min_samples_leaf'] + 1],
    'max_features': [round(max(0.2, best_r['max_features'] - 0.08), 3), round(best_r['max_features'], 3), round(min(1.0, best_r['max_features'] + 0.08), 3)]
}

grid_et = GridSearchCV(
    estimator=ExtraTreesClassifier(random_state=42, n_jobs=-1),
    param_grid=param_grid_et,
    scoring='f1',
    cv=skf,
    n_jobs=-1
)
grid_et.fit(X_train, y_train)
print("Best Parameters from GridSearchCV []:", grid_et.best_params_)
res_grid = evaluate_model(grid_et.best_estimator_, '3. GridSearchCV [] (Final)')

### 5. Final Results Comparison, Model Serialization & Diagnostic Plots
Summarize all three optimization stages, save `best_model1_extratrees.pkl`, and plot the Confusion Matrix and ROC curve.

In [ ]:
results_df = pd.DataFrame([res_base, res_rand, res_grid])
print("\n=== Model 1 (ExtraTreesClassifier) Optimization Progression Table ===")
print(results_df.to_string(index=False))

joblib.dump(grid_et.best_estimator_, 'best_model1_extratrees.pkl')
print("\nSUMMARY_MODEL_1_EXTRATREES:", {'best_params': grid_et.best_params_, 'metrics': res_grid})

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
y_pred_final = grid_et.best_estimator_.predict(X_test)
sns.heatmap(confusion_matrix(y_test, y_pred_final), annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_title('ExtraTrees Confusion Matrix (Test Set)', fontweight='bold')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')
RocCurveDisplay.from_estimator(grid_et.best_estimator_, X_test, y_test, ax=axes[1], name='ExtraTrees (GridSearch)')
axes[1].set_title('ExtraTrees ROC Curve', fontweight='bold')
plt.tight_layout()
plt.savefig('fig_model1_extratrees.png', dpi=300)
plt.show()